<a href="https://colab.research.google.com/github/sebichu/PsygeneAnalyses/blob/Sebastian/BEAM_V1_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# This is a notebook for analyzing BEAM data for the Psygene project.  
<br>
<img src="https://github.com/KravitzLab/KreedLabWiki/blob/main/images/ChatGPT%20Image%20Apr%2020,%202025,%2004_05_24%20PM.png?raw=true" width="300" />

Updated: 11-20-25  
Version: 1.0.0

In [ ]:
# @title Import libraries
import importlib.util
import subprocess
import sys

packages = {
    "ipywidgets": "ipywidgets",
    "pingouin": "pingouin",
    "ipydatagrid": "ipydatagrid",
}

for name, source in packages.items():
    if importlib.util.find_spec(name) is None:
        print(f"Installing {name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", source])

import tempfile
import os
import zipfile
import io
import pandas as pd
from google.colab import files
import re
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FuncFormatter
from statsmodels.formula.api import ols
import statsmodels.api as sm
import ipywidgets as widgets
import warnings
import glob
from ipydatagrid import DataGrid, TextRenderer
from IPython.display import display, clear_output, HTML
from google.colab import files as colab_files
from google.colab import output as colab_output
from google.colab import output
output.enable_custom_widget_manager()
from scipy.optimize import curve_fit
from datetime import datetime
from collections import defaultdict
import matplotlib.dates as mdates
warnings.filterwarnings('ignore')  # this is a bit dangerous but we'll supress all warnings

print("Packages installed.")


In [ ]:
# @title Upload BEAM files


import os, io, zipfile, re
import pandas as pd
from google.colab import files


def upload_BEAM_files(append=False):
    """
    If append=False (default), this replaces global `dataframes` and `loaded_files`.
    If append=True, new files are appended to existing state.
    """
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No files uploaded.")

    # start fresh unless appending
    if not append:
        globals().pop("dataframes", None)
        globals().pop("loaded_files", None)
        globals().pop("device_id", None)

    dfs_existing  = globals().get("dataframes", [])
    names_existing = globals().get("loaded_files", [])

    dfs_new, names_new = [], []

    # continue file_id numbering if appending; otherwise start at 1
    def _next_start():
        try:
            ids = [int(str(getattr(df, "file_id", df.get("file_id", "file_000")))[5:]) for df in dfs_existing]
            return (max(ids) + 1) if ids else 1
        except Exception:
            return 1

    next_id = _next_start() if append else 1

    for name, content in uploaded.items():
        name_str = str(name)

        if name_str.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(content), "r") as z:
                for member in z.namelist():
                    # keep only CSV files
                    if member.endswith("/") or (not member.lower().endswith(".csv")):
                        continue
                    with z.open(member) as f:
                        df = pd.read_csv(f)
                    df["source_file"] = f"{os.path.basename(name_str)}::{member}"
                    df["file_id"] = f"file_{next_id:03d}"
                    dfs_new.append(df)
                    # record the INNER CSV basename (not the ZIP name)
                    names_new.append(os.path.basename(member))
                    next_id += 1

        elif name_str.lower().endswith(".csv"):
            df = pd.read_csv(io.BytesIO(content))
            df["source_file"] = os.path.basename(name_str)
            df["file_id"] = f"file_{next_id:03d}"
            dfs_new.append(df)
            names_new.append(os.path.basename(name_str))
            next_id += 1

        else:
            print(f"[i] Skipping non-CSV file: {name_str}")

    if not dfs_new:
        raise ValueError("No CSV files found. Upload .csv or a .zip containing CSVs.")

    # commit to globals
    dataframes   = (dfs_existing + dfs_new) if append else dfs_new
    loaded_files = (names_existing + names_new) if append else names_new
    globals()["dataframes"] = dataframes
    globals()["loaded_files"] = loaded_files
    device_id = []
    for df in dataframes:
        if "device_id" in df.columns:
            # assume each file is a single device
            dev_val = df["device_id"].iloc[0]
        else:
            dev_val = None  # or raise an error if you want strict behavior
        device_id.append(dev_val)

    globals()["device_id"] = device_id

    print(f"Loaded {len(dfs_new)} CSV(s) from {len(uploaded)} upload(s). "
          f"Current session now has {len(dataframes)} CSV(s).")
    return dataframes, loaded_files

# Use it:
dataframes, loaded_files = upload_BEAM_files(append=False)

In [ ]:
# @title Build Key

# Require that the file-upload cell has already populated these:
assert 'loaded_files' in globals() and 'device_id' in globals(), \
    "Run the 'Upload BEAM files' cell first."

colab_output.enable_custom_widget_manager()

# ---------------------------
# Base: bare-bones Key_Df from loaded data
# ---------------------------
def _make_base_key_df():
    return pd.DataFrame({"filename": loaded_files, "device_id": device_id})

def _file_base(s):
    return os.path.splitext(os.path.basename(str(s)))[0].strip()

def _norm_base_lower(s):
    return _file_base(s).lower()

# ---------------------------
# Key scanner: detect Mouse_ID or filename columns
# ---------------------------
def _scan_key_columns(df):
    """
    Returns dict:
      {
        'has_mouse': bool,
        'has_filename': bool,
        'filename_col': 'filename'|'File'|None,
        'msg': str
      }
    Accepts keys that have either Mouse_ID or a filename column (filename/File).
    """
    info = {'has_mouse': False, 'has_filename': False, 'filename_col': None, 'msg': ''}
    try:
        cols = [str(c).strip() for c in df.columns]
        has_mouse = 'Mouse_ID' in cols
        fname_col = 'filename' if 'filename' in cols else ('File' if 'File' in cols else None)
        info.update({
            'has_mouse': has_mouse,
            'has_filename': fname_col is not None,
            'filename_col': fname_col
        })
        if has_mouse:
            info['msg'] = "'Mouse_ID' found."
        elif fname_col:
            info['msg'] = f"'{fname_col}' found; will match on filename."
        else:
            info['msg'] = "Neither 'Mouse_ID' nor 'filename'/'File' found in provided key."
    except Exception as e:
        info['msg'] = f"Error while checking key: {e}"
    return info

# ---------------------------
# Read uploaded key (CSV/XLSX), accept Mouse_ID or filename
# ---------------------------
def _read_key_from_upload(name, content_bytes):
    """Return (df_or_None, message). Reads CSV/XLSX bytes from Colab upload."""
    ext = name.lower().rsplit('.', 1)[-1] if '.' in name else ''
    try:
        bio = io.BytesIO(content_bytes)
        if ext == 'xlsx':
            xls = pd.ExcelFile(bio, engine='openpyxl')
            frames = [pd.read_excel(xls, sheet_name=s) for s in xls.sheet_names]
            key_df = pd.concat(frames, ignore_index=True, sort=False)
        elif ext == 'csv':
            key_df = pd.read_csv(bio, sep=None, engine='python')
        else:
            return None, f"Unsupported key type .{ext}"

        key_df = key_df.copy()
        key_df.columns = [str(c).strip() for c in key_df.columns]
        scan = _scan_key_columns(key_df)
        if not (scan['has_mouse'] or scan['has_filename']):
            return None, scan['msg']

        # Normalize types/columns we might use later
        if scan['has_mouse']:
            key_df['Mouse_ID'] = key_df['Mouse_ID'].astype(str).str.strip()

        if scan['has_filename']:
            fcol = scan['filename_col']
            key_df[fcol] = key_df[fcol].astype(str).str.strip()
            key_df['_key_file_base_lower'] = key_df[fcol].map(_norm_base_lower)

        # Persist a deterministic copy on disk for reproducibility
        fixed_path = f"_uploaded_key.{ext}"
        with open(fixed_path, "wb") as f:
            f.write(content_bytes)
        globals()['uploaded_key_path'] = fixed_path

        return key_df, f"Key loaded from upload ({name}) and saved to {fixed_path}. {scan['msg']}"
    except Exception as e:
        return None, f"Error reading uploaded key: {e}"

# ---------------------------
# Matching filename <-> Mouse_ID
# ---------------------------
def _match_mouse_id_to_filenames(filenames, key_df):
    """Return DataFrame: filename, Mouse_ID, match_status based on Mouse_ID substring in filename."""
    base_names_lower = [_norm_base_lower(f) for f in filenames]
    mouse_ids = (
        key_df['Mouse_ID']
        .dropna().astype(str).map(str.strip)
        .replace({'': np.nan}).dropna().unique().tolist()
    )
    rows = []
    for fname, base in zip(filenames, base_names_lower):
        hits = [mid for mid in mouse_ids if str(mid).lower() in base]
        if len(hits) == 1:
            rows.append({"filename": fname, "Mouse_ID": hits[0], "match_status": "Matched (Mouse_ID in filename)"})
        elif len(hits) > 1:
            longest = max(len(str(h)) for h in hits)
            best = [h for h in hits if len(str(h)) == longest]
            if len(best) == 1:
                rows.append({"filename": fname, "Mouse_ID": best[0], "match_status": "Matched (longest Mouse_ID token)"})
            else:
                rows.append({"filename": fname, "Mouse_ID": None, "match_status": f"Ambiguous Mouse_ID: {hits}"})
        else:
            rows.append({"filename": fname, "Mouse_ID": None, "match_status": "Mouse_ID not found in filename"})
    return pd.DataFrame(rows)

# ---------------------------
# Build/Rematch function
# ---------------------------
status_box = widgets.Output()
Key_Df = _make_base_key_df()  # start bare-bones

def build_or_rematch_key_df(key_df=None, msg_hint=""):
    """
    If key_df provided and valid:
      - Prefer Mouse_ID mapping if key has Mouse_ID.
      - Else fall back to filename merge (case-insensitive basename).
    Else: keep bare-bones.
    """
    global Key_Df
    files_df = _make_base_key_df().copy()
    files_df['_file_base_lower'] = files_df['filename'].map(_norm_base_lower)

    if key_df is None:
        Key_Df = files_df.drop(columns=['_file_base_lower']).copy()
        Key_Df["match_status"] = "No key"
        with status_box:
            clear_output(wait=True)
            print("Key status: No key provided; showing bare-bones Key_Df.")
        return

    # Identify key capabilities
    scan = _scan_key_columns(key_df)
    kd = key_df.copy()

    # Make a unique version of the key for whichever join we use
    def _dedup(df, subset_cols):
        dup_counts = df[subset_cols].astype(str).agg('|'.join, axis=1).value_counts()
        n_dups = int((dup_counts > 1).sum())
        if n_dups:
            with status_box:
                clear_output(wait=True)
                print(f"Note: {n_dups} duplicate key(s) on {subset_cols}; taking the first occurrence.")
        return df.drop_duplicates(subset=subset_cols, keep="first")

    if scan['has_mouse']:
        # Mouse_ID route (preferred)
        kd['Mouse_ID'] = kd['Mouse_ID'].astype(str).str.strip()
        key_unique = _dedup(kd, ['Mouse_ID'])

        matched = _match_mouse_id_to_filenames(files_df['filename'].tolist(), key_unique)[
            ["filename", "Mouse_ID", "match_status"]
        ]

        Key_Df = (
            files_df
            .merge(matched, on="filename", how="left")
            .merge(key_unique, on="Mouse_ID", how="left", suffixes=("", "_key"))
            .drop(columns=['_file_base_lower'])
        )

    elif scan['has_filename']:
        # Filename route (fallback)
        fcol = scan['filename_col']
        kd['_key_file_base_lower'] = kd[fcol].map(_norm_base_lower)
        key_unique = _dedup(kd, ['_key_file_base_lower'])

        Key_Df = (
            files_df
            .merge(key_unique, left_on="_file_base_lower", right_on="_key_file_base_lower", how="left", suffixes=("", "_key"))
            .drop(columns=['_file_base_lower', '_key_file_base_lower'])
        )
        # Give a simple match_status summary for filename matching
        Key_Df["match_status"] = np.where(
            Key_Df[fcol].notna(), "Matched (filename)", "Filename not found in key"
        )
    else:
        # Neither route available (shouldn't happen due to earlier check)
        Key_Df = files_df.drop(columns=['_file_base_lower']).copy()
        Key_Df["match_status"] = "Key missing Mouse_ID and filename columns"

    with status_box:
        clear_output(wait=True)
        if msg_hint:
            print(msg_hint)
        print(f"Merged key columns into Key_Df ({len(Key_Df)} rows, {len(Key_Df.columns)} cols).")

# ---------------------------
# Grid UI
# ---------------------------
def make_grid(df: pd.DataFrame):
    g = DataGrid(
        df,
        editable=True,
        selection_mode='cell',
        layout={'height': '420px'},
        base_row_size=28,
        base_column_size=120,
    )
    g.default_renderer = TextRenderer(text_wrap=True)
    return g

def rebuild_grid(msg=""):
    global grid, ui
    df = Key_Df.copy().reset_index(drop=True)
    new_grid = make_grid(df)
    ui.children = (upload_row, new_grid, controls, status_box)
    grid = new_grid
    with status_box:
        if msg:
            print(msg)
        print(f"Grid now shows Key_Df ({len(df)} rows, {len(df.columns)} cols)")

# ---------------------------
# Colab-native upload button ONLY (no path UI)
# ---------------------------
upload_btn = widgets.Button(description="Upload", button_style="primary", layout=widgets.Layout(width="120px"))
reset_btn = widgets.Button(description="Reset Key", button_style="warning", layout=widgets.Layout(width="120px"))
download_button = widgets.Button(description='Download', button_style='success', layout=widgets.Layout(width="120px"))

def on_colab_upload(_):
    with status_box:
        clear_output(wait=True)
        print("Opening Colab upload dialog...")
    uploaded = colab_files.upload()  # opens the native Colab picker
    if not uploaded:
        with status_box:
            print("No file selected.")
        return
    name, content = next(iter(uploaded.items()))
    key_df, msg = _read_key_from_upload(name, content)
    if key_df is None:
        build_or_rematch_key_df(None)
        rebuild_grid(f"Key status: {msg}")
    else:
        build_or_rematch_key_df(key_df, msg_hint=f"Key status: {msg}")
        rebuild_grid()

def _load_saved_key_from_disk():
    """Returns (key_df_or_None, message) from uploaded_key_path if present/valid."""
    key_path = globals().get('uploaded_key_path', None)
    if not (key_path and os.path.exists(key_path)):
        return None, "No saved key on disk to reload."
    try:
        ext = key_path.lower().rsplit('.', 1)[-1] if '.' in key_path else ''
        if ext == 'xlsx':
            xls = pd.ExcelFile(key_path, engine='openpyxl')
            frames = [pd.read_excel(xls, sheet_name=s) for s in xls.sheet_names]
            key_df = pd.concat(frames, ignore_index=True, sort=False)
        elif ext == 'csv':
            key_df = pd.read_csv(key_path, sep=None, engine='python')
        else:
            return None, f"Unsupported key type .{ext}"

        key_df = key_df.copy()
        key_df.columns = [str(c).strip() for c in key_df.columns]
        scan = _scan_key_columns(key_df)
        if not (scan['has_mouse'] or scan['has_filename']):
            return None, scan['msg']

        if scan['has_mouse']:
            key_df['Mouse_ID'] = key_df['Mouse_ID'].astype(str).str.strip()
        if scan['has_filename']:
            fcol = scan['filename_col']
            key_df[fcol] = key_df[fcol].astype(str).str.strip()
            key_df['_key_file_base_lower'] = key_df[fcol].map(_norm_base_lower)

        return key_df, f"Key reloaded from {os.path.basename(key_path)}. {scan['msg']}"
    except Exception as e:
        return None, f"Error reading saved uploaded key: {e}"

def on_reset(_):
    """
    Reset now auto-rematches using the last uploaded key if available.
    If no saved key exists or it's invalid, we fall back to bare-bones.
    """
    key_df, msg = _load_saved_key_from_disk()
    if key_df is None:
        build_or_rematch_key_df(None)
        rebuild_grid(f"Key status: {msg} (showing bare-bones Key_Df).")
    else:
        build_or_rematch_key_df(key_df, msg_hint=f"Key status: {msg}")
        rebuild_grid("Reset: reloaded saved key and rematched.")

def download_df(_):
    global Key_Df
    with status_box:
        clear_output(wait=True)
        print("Saving latest Key_Df as XLSX ...")
    try:
        path = "/content/Key_Df.xlsx"
        Key_Df.to_excel(path, index=False, engine='openpyxl')
        colab_files.download(path)
        with status_box:
            clear_output(wait=True)
            print("Saved and downloading Key_Df.xlsx ...")
    except Exception as e:
        with status_box:
            clear_output(wait=True)
            print(f"Error while saving/downloading: {e}")

upload_btn.on_click(on_colab_upload)
reset_btn.on_click(on_reset)
download_button.on_click(download_df)

upload_row = widgets.HBox([
    widgets.HTML("<b>Optional key:</b>"),
    upload_btn,
    reset_btn,
    download_button
])

# ---------------------------
# Edit / rematch / download controls
# ---------------------------
new_col_name    = widgets.Text(placeholder='Enter new column name', description='New Col:')
add_col_button  = widgets.Button(description='Add Column', button_style='info')
apply_button    = widgets.Button(description='Apply Changes', button_style='primary', layout=widgets.Layout(width="120px"))

def add_column(_):
    global Key_Df
    col = new_col_name.value.strip()
    with status_box:
        clear_output(wait=True)
        if not col:
            print("Please enter a column name."); return
        if col in Key_Df.columns:
            print(f"Column '{col}' already exists."); return
        Key_Df[col] = ""
        print(f"Added column '{col}' to Key_Df.")
    rebuild_grid()

def apply_edits(_):
    global Key_Df
    try:
        Key_Df = grid.data.copy().reset_index(drop=True)
        with status_box:
            clear_output(wait=True)
            print(f"Applied grid edits to Key_Df ({len(Key_Df)} rows, {len(Key_Df.columns)} cols).")
    except Exception as e:
        with status_box:
            clear_output(wait=True)
            print(f"Error applying edits: {e}")

add_col_button.on_click(add_column)
apply_button.on_click(apply_edits)

controls = widgets.HBox([new_col_name, add_col_button, apply_button])

# ---------------------------
# Initialize UI
# ---------------------------
Key_Df = _make_base_key_df()
Key_Df["match_status"] = "No key"

grid = make_grid(Key_Df.copy().reset_index(drop=True))
ui = widgets.VBox([upload_row, grid, controls, status_box])
display(ui)

In [ ]:
#@title Individual plots
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
import ipywidgets as widgets
from IPython.display import display, clear_output

assert 'dataframes' in globals() and 'loaded_files' in globals(), \
    "Run the 'Upload BEAM files' cell first."

# List of files we’ll index into with the slider
files_list = [os.path.basename(str(f)) for f in loaded_files]
N = len(files_list)

# ----- UI -----
idx_slider = widgets.IntSlider(
    min=0,
    max=max(0, N-1),
    step=1,
    value=0,
    description='File index',
    continuous_update=True,
)
status_lbl = widgets.HTML()
out = widgets.Output()

def plot_one_file(idx):
    """Plot activity_percent vs datetime for file at index idx."""
    df = dataframes[idx].copy()
    fname = files_list[idx]

    # sanity checks
    if 'datetime' not in df.columns:
        raise KeyError(f"'datetime' column not found in file {loaded_files[idx]}")
    if 'activity_percent' not in df.columns:
        raise KeyError(f"'activity_percent' column not found in file {loaded_files[idx]}")

    # ensure datetime dtype
    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
    df = df.dropna(subset=['datetime'])
    df = df.sort_values('datetime')

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(df['datetime'], df['activity_percent'])
    ax.set_xlabel('Time of day')
    ax.set_ylabel('activity_percent')
    ax.set_title(f"{fname}")
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    # X axis: only 12am, 6 am, 12pm, 6 pm
    locator = mdates.HourLocator(byhour=[0, 6, 12, 18])
    ax.xaxis.set_major_locator(locator)

    def _fmt_lower_ampm(x, pos=None):
        dt = mdates.num2date(x)
        h = dt.hour
        if h == 0:
            return "12am"
        elif h == 6:
            return "6 am"
        elif h == 12:
            return "12pm"
        elif h == 18:
            return "6pm"
        else:
            return ""  # hide any stray ticks

    ax.xaxis.set_major_formatter(FuncFormatter(_fmt_lower_ampm))

    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

def on_slider_change(change):
    idx = change["new"]
    with out:
        clear_output(wait=True)
        status_lbl.value = f"<b>File {idx+1}/{N}:</b> {files_list[idx]}"
        plot_one_file(idx)

# hook up the slider
idx_slider.observe(on_slider_change, names='value')

# initial draw
with out:
    clear_output(wait=True)
    status_lbl.value = f"<b>File 1/{N}:</b> {files_list[0]}"
    plot_one_file(0)

ui = widgets.VBox([idx_slider, status_lbl, out])
display(ui)

In [ ]:
# @title Analyze BEAM metrics
import os, re
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
import ipywidgets as widgets
from IPython.display import display
from google.colab import files as colab_files

"""
This code requires Wt or control in the genotype and Gene.
For normalisation, genotypes are normalised to the WT Z-score of their own Gene.
"""

# ---------- config & globals ----------
OMEGA = 2 * np.pi / 24.0
LIGHTS_OFF_HOUR = 18  # ZT0 at 18:00 (6 pm)
fit_storage = []

ACTIVITY_COL_CANDS = [
    "activity_percent", "Activity_percent", "Activity %", "activity%", "activity", "Activity"
]

# ---------- helpers ----------
def cosinor(t, mesor, amplitude, acrophase):
    t = np.asarray(t, dtype=float)
    return mesor + amplitude * np.cos(OMEGA * t + acrophase)

def clock_to_zt_unsigned(hours, lights_off_hour=LIGHTS_OFF_HOUR):
    """Clock hour(s) [0..23] -> ZT [0..24), ZT0 at lights-off."""
    return (np.asarray(hours, dtype=float) - lights_off_hour) % 24

def zt_unsigned_to_signed(zt_hours):
    """ZT [0..24) -> signed ZT [-12, +12)."""
    zt = np.asarray(zt_hours, dtype=float)
    return ((zt + 12) % 24) - 12

def find_col(df, cands):
    """Case/format-insensitive column finder."""
    norm = lambda s: "".join(ch for ch in str(s).lower() if ch.isalnum())
    cols_norm = {norm(c): c for c in df.columns}
    for cand in cands:
        k = norm(cand)
        if k in cols_norm:
            return cols_norm[k]
    return None

def _inner_basename(s):
    # "zip.zip::INNER.csv" or ".../INNER.csv" -> "INNER.csv"
    p = re.split(r"::|:", str(s))[-1]
    return os.path.basename(p.replace("\\", "/"))

# ---------- build BEAM_data from Key_Df (no remapping here) ----------
def build_BEAM_data_from_key(raw_list, loaded_files, key_df):
    """
    raw_list: list of BEAM dataframes (from upload_BEAM_files)
    loaded_files: list of filenames (same order as raw_list)
    key_df: Key_Df, already matched to Mouse_ID, Gene, Genotype, etc.
    """
    if not raw_list:
        raise RuntimeError("No BEAM CSVs loaded.")
    if len(raw_list) != len(loaded_files):
        raise RuntimeError("dataframes and loaded_files lengths do not match.")

    # 1) concat and tag with filename/file_key
    tagged = []
    for df, fname in zip(raw_list, loaded_files):
        d = df.copy()
        fname_base = os.path.basename(str(fname))
        d["filename"]   = fname_base
        d["file_key"]   = fname_base   # use filename as file_key
        d["file_label"] = fname_base
        tagged.append(d)

    raw = pd.concat(tagged, ignore_index=True)

    # 2) ensure datetime remains a COLUMN
    if "datetime" not in raw.columns:
        raise RuntimeError("BEAM CSVs must contain a 'datetime' column.")
    raw["datetime"] = pd.to_datetime(raw["datetime"], errors="coerce")
    raw = raw[raw["datetime"].notna()].copy()
    raw = raw.sort_values("datetime")

    # 3) prepare key for merge (use basename of filename)
    md = key_df.copy()
    if "filename" not in md.columns:
        raise RuntimeError("Key_Df must contain a 'filename' column.")
    md["filename"] = md["filename"].astype(str).map(os.path.basename)

    # 4) merge all key metadata directly; no Mouse_ID remapping here
    raw = raw.merge(md, on="filename", how="left", suffixes=("", "_key"))

    # 5) basic sanity: Mouse_ID and Gene should exist after merge
    if "Mouse_ID" not in raw.columns:
        raise RuntimeError("Key_Df must contain 'Mouse_ID' column (and it must match filenames).")
    if find_col(raw, ("Gene", "gene", "GENE")) is None:
        raise RuntimeError("Key_Df must contain a 'Gene' column for per-Gene WT normalization.")

    return raw

# ---------- make sure we have Key_Df + uploaded data ----------
assert "Key_Df" in globals() and isinstance(Key_Df, pd.DataFrame), "Build/rematch Key_Df first."
assert "dataframes" in globals() and "loaded_files" in globals(), "Run the 'Upload BEAM files' cell first."

metadata_df = Key_Df.copy().reset_index(drop=True)
BEAM_data = build_BEAM_data_from_key(dataframes, loaded_files, metadata_df)

# ---------- find activity column & compute WT-referenced z (per Gene) ----------
activity_col = find_col(BEAM_data, ACTIVITY_COL_CANDS)
if activity_col is None:
    raise RuntimeError(f"Could not find activity column among: {ACTIVITY_COL_CANDS}")

# Coerce numeric; strip '%' if present
if (
    BEAM_data[activity_col].dtype == object
    and BEAM_data[activity_col].astype(str).str.contains("%").any()
):
    BEAM_data[activity_col] = (
        BEAM_data[activity_col]
        .astype(str)
        .str.replace("%", "", regex=False)
    )
BEAM_data[activity_col] = pd.to_numeric(BEAM_data[activity_col], errors="coerce")

# Gene column from merged key
gene_col = find_col(BEAM_data, ("Gene", "gene", "GENE"))
if gene_col != "Gene":
    BEAM_data.rename(columns={gene_col: "Gene"}, inplace=True)

# Genotype column from merged key (includes case like 'Genotype$')
geno_col = find_col(
    BEAM_data,
    ("Genotype$", "Genotype", "genotype", "GENOTYPE", "Gt", "GT", "geno")
)
if geno_col is None:
    raise RuntimeError("Key_Df / BEAM_data must contain a Genotype column (e.g. 'Genotype$').")

# Simple genotype normalization: Wt, Het, Hom, Hemi
def norm_genotype(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower().replace(" ", "")
    if s == "wt":
        return "WT"
    if s == "WT; -VE":
        return "WT"
    if s == "het":
        return "Het"
    if s == "hom":
        return "Hom"
    if s == "hemi":
        return "Hemi"
    # fallback: keep original string so you can see odd entries later
    return str(x)

BEAM_data["Genotype_raw"] = BEAM_data[geno_col]
BEAM_data["Genotype_norm"] = BEAM_data["Genotype_raw"].map(norm_genotype)

# WT rows per Gene (Wt is the only reference)
wt_df = BEAM_data[BEAM_data["Genotype_norm"] == "WT"].copy()
if wt_df.empty:
    raise RuntimeError("No Wt rows found in BEAM_data; cannot form per-Gene WT references.")

# WT mean and SD within each Gene
gene_stats = (
    wt_df
    .groupby("Gene")[activity_col]
    .agg(["mean", "std"])
    .rename(columns={"mean": "mu_geneWT", "std": "sd_geneWT"})
)

# Attach Gene-specific WT stats back to all rows
BEAM_data = BEAM_data.merge(gene_stats, on="Gene", how="left")

# Warn for Genes with no WT (they'll get NaN z-scores)
missing_ref_genes = BEAM_data.loc[BEAM_data["mu_geneWT"].isna(), "Gene"].dropna().unique()
if len(missing_ref_genes) > 0:
    print(
        "Warning: these Genes have no Wt animals and will have NaN z_WTref:",
        list(missing_ref_genes),
    )

# Avoid divide-by-zero
BEAM_data.loc[BEAM_data["sd_geneWT"] == 0, "sd_geneWT"] = np.nan

# Final per-Gene WT-referenced z-score (per-row)
BEAM_data["z_WTref"] = (
    BEAM_data[activity_col] - BEAM_data["mu_geneWT"]
) / BEAM_data["sd_geneWT"]

# ---------- per-file raw mean activity (no z-score) ----------
mean_activity_df = (
    BEAM_data
    .dropna(subset=[activity_col])
    .groupby("filename")[activity_col]
    .mean()
    .reset_index()
    .rename(columns={activity_col: "Mean_activity"})
)

# ---------- Night_z and Day_z per file (before plotting) ----------
# Night: ZT in [0, 12)
# Day:   ZT in [12, 24)

tmp = BEAM_data[["file_key", "filename", "datetime", "z_WTref"]].copy()
tmp["datetime"] = pd.to_datetime(tmp["datetime"], errors="coerce")
tmp = tmp.dropna(subset=["datetime", "z_WTref"])

clock_hour = tmp["datetime"].dt.hour.astype(float)
zt_hour = (clock_hour - LIGHTS_OFF_HOUR) % 24

tmp["phase"] = np.where(zt_hour < 12, "Night", "Day")

phase_agg = (
    tmp.groupby(["file_key", "filename", "phase"])["z_WTref"]
       .mean()
       .unstack("phase")
       .rename(columns={"Night": "Night_z", "Day": "Day_z"})
       .reset_index()
)

# ---------- cosinor per file (ZT-based) ----------
def cosinor_per_file_on_zscore(data, metric_col="z_WTref", min_points=4):
    """
    Cosinor per file_key, using Mouse_ID from Key_Df (already merged).
    Fits in ZT where ZT0 = LIGHTS_OFF_HOUR.
    Uses the 'datetime' column explicitly.

    Amplitude_z in the output is **peak-to-trough range**
    (max fitted z – min fitted z) for the cosinor curve.
    """
    if "file_key" not in data.columns:
        raise RuntimeError("BEAM_data must contain 'file_key' (built from filename).")
    if "datetime" not in data.columns:
        raise RuntimeError("BEAM_data must contain a 'datetime' column (not just as index).")

    label_col = "file_label" if "file_label" in data.columns else "file_key"

    df_all = data.copy()
    # enforce datetime column type
    df_all["datetime"] = pd.to_datetime(df_all["datetime"], errors="coerce")
    df_all = df_all[df_all["datetime"].notna()].copy()
    df_all = df_all.sort_values("datetime")

    results = []
    global fit_storage
    fit_storage = []  # clear every run

    for file_key, df_file in df_all.groupby("file_key"):
        df_file = df_file.copy()
        file_label = df_file[label_col].iloc[0]

        # ---- Mouse_ID from merged key ----
        if "Mouse_ID" in df_file.columns:
            mid_series = (
                df_file["Mouse_ID"]
                .astype(str).str.strip()
                .replace("", np.nan)
                .dropna()
            )
        else:
            mid_series = pd.Series([], dtype=object)

        if len(mid_series) == 0:
            mouse_id = ""
            id_source = "unknown"
            mixed_ids = ""
        else:
            counts = mid_series.value_counts()
            mouse_id = counts.index[0]
            id_source = "from_Key_Df"
            mixed_ids = ";".join(counts.index.tolist()) if len(counts) > 1 else mouse_id

        # ---- ZT hour from datetime column ----
        dt_series = df_file["datetime"]
        zt_hr = ((dt_series.dt.hour - LIGHTS_OFF_HOUR) % 24).astype(float)

        df_file = df_file.assign(ZT_hr=zt_hr)
        s = (
            df_file.loc[df_file["ZT_hr"].notna()]
                   .groupby("ZT_hr", dropna=True)[metric_col]
                   .mean()
                   .astype(float)
        )

        t_zt = s.index.to_numpy(dtype=float)  # ZT hours [0..23]
        y = s.values
        m = np.isfinite(y)
        t_zt = t_zt[m]
        y = y[m]

        mesor = amp_param = amp_range = acrophase = r2 = np.nan
        status = "insufficient_points"
        if t_zt.size >= min_points and np.nanstd(y) > 0:
            guess = [np.nanmean(y), (np.nanmax(y) - np.nanmin(y)) / 2, 0.0]
            try:
                params, _ = curve_fit(
                    cosinor, t_zt, y, p0=guess, maxfev=20000
                )
                mesor, amp_param, acrophase = map(float, params)
                if amp_param < 0:
                    amp_param = -amp_param
                    acrophase += np.pi
                # fitted curve at observed times
                y_fit = cosinor(t_zt, mesor, amp_param, acrophase)
                # amplitude as peak-to-trough range on the fitted curve
                amp_range = float(np.nanmax(y_fit) - np.nanmin(y_fit))

                ss_res = float(np.nansum((y - y_fit) ** 2))
                ss_tot = float(np.nansum((y - np.nanmean(y)) ** 2))
                r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else np.nan
                status = "ok"
            except Exception:
                status = "fit_failed"

        # Peak in ZT
        peak_zt_unsigned = float(((-acrophase) % (2 * np.pi)) / OMEGA) if np.isfinite(acrophase) else np.nan
        peak_zt_signed   = float(zt_unsigned_to_signed(peak_zt_unsigned)) if np.isfinite(peak_zt_unsigned) else np.nan
        peak_clock_hr    = float((peak_zt_unsigned + LIGHTS_OFF_HOUR) % 24) if np.isfinite(peak_zt_unsigned) else np.nan

        results.append({
            "file": file_label,
            "file_key": file_key,
            "Mouse_ID": mouse_id,
            "ID_source": id_source,
            "Mouse_IDs_seen_in_file": mixed_ids if id_source == "from_Key_Df" else "",
            "MESOR_z": mesor,
            "Amplitude_z": amp_range,
            "Acrophase_ZT_signed":   peak_zt_signed,    # [-12, +12)
            "Cosinor_R2": r2,
            "N_hours_used": int(t_zt.size),
            "Fit_Status": status,
        })

        fit_storage.append({
            "file": file_label,
            "file_key": file_key,
            "Mouse_ID": mouse_id,
            "t_zt_unsigned": t_zt,
            "y_z": y,
            "fit_params": (mesor, amp_param, acrophase),
            "fit_peak_zt_unsigned": peak_zt_unsigned,
            "r_squared": r2,
            "status": status,
            "id_source": id_source
        })

    return pd.DataFrame(results)

# ---------- plotting ----------
def plot_file_fit(index):
    d = fit_storage[index]
    file_label = d["file"]
    t_zt_unsigned = d["t_zt_unsigned"]
    y = d["y_z"]
    mesor, amplitude, acrophase = d["fit_params"]
    peak_zt_unsigned = d["fit_peak_zt_unsigned"]
    r2 = d["r_squared"]
    status = d["status"]

    # Convert x-values to signed ZT for a centered plot
    x_obs = zt_unsigned_to_signed(t_zt_unsigned)
    order_obs = np.argsort(x_obs)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(x_obs[order_obs], y[order_obs], "o", label="Observed hourly z")

    if np.isfinite(mesor) and np.isfinite(amplitude) and np.isfinite(acrophase):
        t_fit_zt = np.linspace(0, 23.999, 1000)
        y_fit = cosinor(t_fit_zt, mesor, amplitude, acrophase)
        x_fit = zt_unsigned_to_signed(t_fit_zt)
        order_fit = np.argsort(x_fit)
        ax.plot(x_fit[order_fit], y_fit[order_fit], "-", label=f"Cosinor ({status})")

        if np.isfinite(peak_zt_unsigned):
            ax.axvline(
                zt_unsigned_to_signed(peak_zt_unsigned),
                linestyle="--",
                label="Acrophase (ZT)",
            )

    ax.set_title(f"{file_label}  |  ZT0=lights off ({LIGHTS_OFF_HOUR:02d}:00)")
    ax.set_xlabel("ZT (hours)   [−12 … +12; ZT0 at lights off]")
    ax.set_ylabel("Z of Activity")
    ax.set_xlim(-12, 12)
    ax.set_xticks([-12, -6, 0, 6, 12])
    sns.despine()
    ax.legend(frameon=False)
    plt.show()

def show_interactive_file_cosinor():
    if len(fit_storage) == 0:
        print("No fits yet; run cosinor_per_file_on_zscore first.")
        return
    slider = widgets.IntSlider(
        min=0, max=len(fit_storage) - 1, step=1, description="File idx"
    )
    widgets.interact(plot_file_fit, index=slider)

# ---------- run cosinor ----------
cosinor_df = cosinor_per_file_on_zscore(BEAM_data, metric_col="z_WTref")
print(cosinor_df["Fit_Status"].value_counts(dropna=False))
show_interactive_file_cosinor()

# ---------- Build BEAM_metrics (Key_Df + cosinor + Night/Day z + raw activity) ----------
# 1. Make a copy of cosinor_df and align filename column name with Key_Df
metrics_df = cosinor_df.copy()
if "file" not in metrics_df.columns:
    raise RuntimeError("cosinor_df does not have a 'file' column; check the cosinor code.")
metrics_df["filename"] = metrics_df["file"].astype(str).map(os.path.basename)

# 2. Make sure Key_Df has a normalized filename column as well
key_df = Key_Df.copy()
if "filename" not in key_df.columns:
    raise RuntimeError("Key_Df must contain a 'filename' column.")
key_df["filename"] = key_df["filename"].astype(str).map(os.path.basename)

# 3. Merge: one row per file with all key metadata + cosinor metrics
BEAM_metrics = key_df.merge(
    metrics_df.drop(columns=["file"]),  # drop old 'file' to avoid confusion
    on="filename",
    how="left",                         # keep all files from Key_Df
    suffixes=("", "_cosinor")
)

# 4. Merge Night_z and Day_z
BEAM_metrics = BEAM_metrics.merge(
    phase_agg[["filename", "Night_z", "Day_z"]],
    on="filename",
    how="left"
)

# 5. Merge raw mean activity (no z-score)
BEAM_metrics = BEAM_metrics.merge(
    mean_activity_df,
    on="filename",
    how="left"
)

# 6. Drop plumbing columns we don't want in the output
cols_to_drop = [
    "file_key",
    "ID_source",
    "Mouse_IDs_seen_in_file",
    "Fit_Status",
    "N_hours_used",
]
drop_existing = [c for c in cols_to_drop if c in BEAM_metrics.columns]
BEAM_metrics = BEAM_metrics.drop(columns=drop_existing)

# 7. Save to Excel and download
out_path = "/content/BEAM_metrics.xlsx"
BEAM_metrics.to_excel(out_path, index=False, engine="openpyxl")
colab_files.download(out_path)

print(f"Saved and downloading: {out_path}")
print(f"Rows: {len(BEAM_metrics)}, columns: {len(BEAM_metrics.columns)}")


In [ ]:
# @title Group for plotting

import os
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- sanity ---
if 'metadata_df' not in globals() or metadata_df is None or metadata_df.empty:
    raise RuntimeError("metadata_df is missing or empty. Build metadata_df (copy of Key_Df) first.")

EXCLUDE_LOWER = {"match_status"}   # everything else is allowed

def _build_file_column(df):
    if "filename" in df.columns:
        return df["filename"].apply(lambda p: os.path.basename(str(p)))
    if "FED3_from_file" in df.columns and "Date_from_file" in df.columns:
        return "FED" + df["FED3_from_file"].astype(str) + "_" + df["Date_from_file"].astype(str)
    if "FED3_from_file" in df.columns:
        return "FED" + df["FED3_from_file"].astype(str)
    return df.index.astype(str)

def _norm_val(x):
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none"}:
        return "UNK"
    return s.upper()

def _build_group_row(row, ordered_cols):
    if not ordered_cols:
        return "ALL"
    return " | ".join(_norm_val(row[c]) for c in ordered_cols)

def build_mapping(ordered_cols):
    _meta = metadata_df.copy()
    _meta["filename"] = _build_file_column(_meta)
    _meta["Group"] = _meta.apply(lambda r: _build_group_row(r, ordered_cols), axis=1)
    mapping = (
        _meta[["filename", "Group"]]
        .dropna(subset=["filename"])
        .drop_duplicates()
        .sort_values(["Group", "filename"])
        .reset_index(drop=True)
    )
    return mapping

def _unique_keep_order(seq):
    seen = set(); out = []
    for x in seq:
        if x not in seen:
            seen.add(x); out.append(x)
    return out

# ---------- UI (fixed sizes + grid) ----------
PX_W = "260px"   # list box width
PX_H = "160px"   # list box height
BTN_W = "160px"  # button column width
HDR_H = "28px"   # header cell height (consistent across all headers)

title = widgets.HTML("<h3>Select columns to group by for X and Hue, then reorder X to set hierarchy</h3>")

all_cols = sorted((c for c in metadata_df.columns if str(c).lower() not in EXCLUDE_LOWER), key=str.lower)

def header(text):
    return widgets.HTML(
        f"<div style='height:{HDR_H};display:flex;align-items:flex-end;'>"
        f"<h4 style=\"margin:0;\">{text}</h4></div>"
    )

# Headers (row 1 of grid)
available_hdr = header("Available")
actions_hdr   = header("Actions")
x_hdr         = header("X grouping")
hue_hdr       = header("Hue grouping")

# Widgets (row 2 of grid)
available = widgets.SelectMultiple(
    options=all_cols, value=tuple(), rows=14,
    layout=widgets.Layout(
        width=PX_W, height=PX_H, min_width=PX_W, max_width=PX_W,
        min_height=PX_H, max_height=PX_H, flex="0 0 auto"
    )
)

right_x = widgets.Select(
    options=[], value=None, rows=8,
    layout=widgets.Layout(
        width=PX_W, height=PX_H, min_width=PX_W, max_width=PX_W,
        min_height=PX_H, max_height=PX_H, flex="0 0 auto"
    )
)

right_hue = widgets.Select(
    options=[], value=None, rows=8,
    layout=widgets.Layout(
        width=PX_W, height=PX_H, min_width=PX_W, max_width=PX_W,
        min_height=PX_H, max_height=PX_H, flex="0 0 auto"
    )
)

# Buttons
btn_add_x    = widgets.Button(description="Add to X ▶", button_style='primary', layout=widgets.Layout(width=BTN_W))
btn_add_hue  = widgets.Button(description="Add to Hue ▶",button_style='primary', layout=widgets.Layout(width=BTN_W))
btn_clear    = widgets.Button(description="Clear", button_style='danger', layout=widgets.Layout(width=BTN_W))
btn_up       = widgets.Button(description="↑ Up (X only)", layout=widgets.Layout(width=BTN_W))
btn_down     = widgets.Button(description="↓ Down (X only)", layout=widgets.Layout(width=BTN_W))

controls_col = widgets.VBox(
    [btn_add_x, btn_add_hue, btn_clear, btn_up, btn_down],
    layout=widgets.Layout(
        align_items="center",
        width=BTN_W, min_width=BTN_W, max_width=BTN_W,
        height=PX_H, min_height=PX_H, max_height=PX_H,
        flex="0 0 auto"
    )
)

btn_build = widgets.Button(description="Build Groups", button_style='success', layout=widgets.Layout(width="160px"))

# IMPORTANT: rename from `output` to `out_box`
out_box = widgets.Output()

# --- Callbacks ---
def on_add_x(_):
    sel = list(available.value)
    if not sel: return
    new_opts = _unique_keep_order(list(right_x.options) + sel)
    right_x.value = None
    right_x.options = new_opts
    right_x.value = new_opts[-1] if new_opts else None

def on_add_hue(_):
    sel = list(available.value)
    if not sel: return
    new_opts = _unique_keep_order(list(right_hue.options) + sel)
    right_hue.value = None
    right_hue.options = new_opts
    right_hue.value = new_opts[-1] if new_opts else None

def on_clear(_):
    right_x.value = None; right_x.options = []
    right_hue.value = None; right_hue.options = []

def on_up(_):
    item = right_x.value
    if item is None: return
    opts = list(right_x.options)
    i = opts.index(item)
    if i > 0:
        opts[i-1], opts[i] = opts[i], opts[i-1]
        right_x.value = None; right_x.options = opts; right_x.value = item

def on_down(_):
    item = right_x.value
    if item is None: return
    opts = list(right_x.options)
    i = opts.index(item)
    if i < len(opts) - 1:
        opts[i+1], opts[i] = opts[i], opts[i+1]
        right_x.value = None; right_x.options = opts; right_x.value = item

def on_build(_):
    # use out_box instead of output
    with out_box:
        clear_output()
        ordered_cols_x = list(right_x.options)
        ordered_cols_hue = list(right_hue.options)

        mapping_x = build_mapping(ordered_cols_x)
        mapping_hue = build_mapping(ordered_cols_hue)

        _meta = metadata_df.copy()
        _meta["filename"] = _build_file_column(_meta)
        _meta["XGroup"] = _meta.apply(lambda r: _build_group_row(r, ordered_cols_x), axis=1)
        _meta["HueGroup"] = _meta.apply(lambda r: _build_group_row(r, ordered_cols_hue), axis=1)
        mapping_both = (
            _meta[["filename", "XGroup", "HueGroup"]]
            .dropna(subset=["filename"])
            .drop_duplicates()
            .sort_values(["XGroup", "HueGroup", "filename"])
            .reset_index(drop=True)
        )

        globals()['files_to_group_x'] = mapping_x.copy()
        globals()['files_to_group_hue'] = mapping_hue.copy()
        globals()['files_to_group_both'] = mapping_both.copy()
        globals()['selected_group_cols_x'] = ordered_cols_x.copy()
        globals()['selected_group_cols_hue'] = ordered_cols_hue.copy()

        print("X-axis grouping (hierarchy):", ordered_cols_x if ordered_cols_x else ["ALL"])
        print(f"Total unique files (X map): {mapping_x['filename'].nunique()}")
        display(widgets.HTML("<b>X-group summary</b>"))
        display((mapping_x.groupby("Group", dropna=False)["filename"]
                 .nunique().sort_values(ascending=False)
                 .rename("UniqueFiles").to_frame()))

        print("\nHue grouping:", ordered_cols_hue if ordered_cols_hue else ["ALL"])
        print(f"Total unique files (Hue map): {mapping_hue['filename'].nunique()}")
        display(widgets.HTML("<b>Hue-group summary</b>"))
        display((mapping_hue.groupby("Group", dropna=False)["filename"]
                 .nunique().sort_values(ascending=False)
                 .rename("UniqueFiles").to_frame()))
        print("\nCombined mapping available as `files_to_group_both` (filename, XGroup, HueGroup)")

# Wire up
btn_add_x.on_click(on_add_x)
btn_add_hue.on_click(on_add_hue)
btn_clear.on_click(on_clear)
btn_up.on_click(on_up)
btn_down.on_click(on_down)
btn_build.on_click(on_build)

# ----- Grid layout -----
grid = widgets.GridBox(
    children=[
        available_hdr, actions_hdr, x_hdr, hue_hdr,     # row 1: headers
        available,     controls_col, right_x, right_hue # row 2: widgets
    ],
    layout=widgets.Layout(
        grid_template_columns=f"{PX_W} {BTN_W} {PX_W} {PX_W}",
        grid_template_rows="auto auto",
        grid_gap="6px 16px",
        align_items="flex-start",
        justify_items="flex-start",
        width="100%"
    )
)

ui = widgets.VBox([title, grid, widgets.HBox([btn_build]), out_box])
display(ui)

In [ ]:
# @title Plot BEAM metrics!

import os, time, shutil, re, itertools
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Stats
import pingouin as pg
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Optional Colab download
try:
    from google.colab import files as colab_files
except Exception:
    colab_files = None

ALPHA = 0.6  # apply to both bars and dots

# -----------------------
# 0) Preconditions & source (BEAM metrics)
# -----------------------
if "BEAM_metrics" in globals() and BEAM_metrics is not None and not BEAM_metrics.empty:
    bm = BEAM_metrics.copy()
elif "BEAM_metrics_csv" in globals() and BEAM_metrics_csv is not None and not BEAM_metrics_csv.empty:
    bm = BEAM_metrics_csv.copy()
else:
    raise RuntimeError("No BEAM metrics table found. Run the BEAM metrics cell first.")

if "filename" not in bm.columns:
    if "File" in bm.columns:
        bm["filename"] = bm["File"].astype(str)
    else:
        raise RuntimeError("BEAM metrics table must include a 'filename' column.")

# -----------------------
# Merge in XGroup/HueGroup from grouping widget
# -----------------------
def _basename_col(s):
    return os.path.basename(str(s))

def _src_name(df):
    if "filename" in df.columns:
        return "filename"
    if "File" in df.columns:
        return "File"
    return None

if ("XGroup" not in bm.columns) or ("HueGroup" not in bm.columns):
    if "files_to_group_both" in globals() and files_to_group_both is not None and not files_to_group_both.empty:
        m = files_to_group_both.copy()
        m_src = _src_name(m)
        if m_src is None:
            raise RuntimeError("Grouping table must include 'filename' (or legacy 'File').")
        m["file_base"]  = m[m_src].apply(_basename_col)
        bm["file_base"] = bm["filename"].apply(_basename_col)
        bm = bm.merge(m[["file_base","XGroup","HueGroup"]], on="file_base", how="left").drop(columns=["file_base"])
        bm["XGroup"]   = bm["XGroup"].fillna("UNASSIGNED")
        bm["HueGroup"] = bm["HueGroup"].fillna("UNASSIGNED")
    elif "files_to_group" in globals() and files_to_group is not None and not files_to_group.empty:
        m = files_to_group.copy()
        m_src = _src_name(m)
        if m_src is None:
            raise RuntimeError("Grouping table must include 'filename' (or legacy 'File').")
        m["file_base"]  = m[m_src].apply(_basename_col)
        bm["file_base"] = bm["filename"].apply(_basename_col)
        bm = bm.merge(m[["file_base","Group"]], on="file_base", how="left").drop(columns=["file_base"])
        bm["Group"] = bm["Group"].fillna("UNASSIGNED")
        bm["XGroup"] = bm["Group"]
        bm["HueGroup"] = "ALL"
    else:
        raise RuntimeError("Missing X/Hue mapping. Run the grouping widget (Build Groups) first.")

# -----------------------
# 1) Melt to long format
# -----------------------

# Metrics we care about in BEAM_metrics
base_metric_names = [
    "MESOR_z",
    "Amplitude_z",
    "Acrophase_ZT_signed",
    "Cosinor_R2",
    "Night_z",
    "Day_z",
    "Mean_activity",
]

metric_cols = []
for c in bm.columns:
    if pd.api.types.is_numeric_dtype(bm[c]):
        for base in base_metric_names:
            if c == base or c.startswith(base + "_"):
                metric_cols.append(c)
                break
seen = set()
metric_cols = [c for c in metric_cols if not (c in seen or seen.add(c))]
if not metric_cols:
    raise RuntimeError("No numeric BEAM metric columns found among expected BEAM metrics.")

candidate_id_vars = [
    "Genotype","Genotype_norm","Sex","Strain","Gene","Start_Date",
    "Mouse_ID","Session_type","XGroup","HueGroup"
]
id_vars = [c for c in candidate_id_vars if c in bm.columns]
for need in ["XGroup","HueGroup","filename"]:
    if need not in id_vars:
        id_vars.append(need)

long_df = pd.melt(
    bm,
    id_vars=id_vars,
    value_vars=metric_cols,
    var_name="variable",
    value_name="value"
)
bm.head()

# -----------------------
# 2) Ordering helpers
# -----------------------

# Define hue order priority
HUE_PRIORITY = ["Female", "Male", "F", "M", "Day", "Night", "Light", "Dark", "ALL", "UNASSIGNED"]

def _is_wt_group(g):
    u = str(g).strip().upper()
    tokens = [t for t in re.split(r'[^A-Z0-9]+', u) if t]
    WT_ALIASES = {"WT", "WILDTYPE", "CONTROL", "CTRL"}
    return any(t in WT_ALIASES for t in tokens)

def _is_unassigned_token(s):
    return (str(s).strip().upper() in {"", "UNASSIGNED", "NONE", "NA", "N/A"})

def _x_levels(xname):
    s = str(xname)
    parts = [p.strip() for p in s.split("|")]
    wanted = globals().get("selected_group_cols_x", None)
    if isinstance(wanted, (list, tuple)) and wanted:
        if len(parts) < len(wanted):
            parts += [""] * (len(wanted) - len(parts))
        else:
            parts = parts[:len(wanted)]
    return parts

def _hier_sort_key(g):
    lv = _x_levels(g)
    norm = []
    for tok in lv:
        is_blank = 1 if _is_unassigned_token(tok) else 0
        norm.append((is_blank, str(tok).upper()))
    wt_present = any(_is_wt_group(tok) for tok in lv) or _is_wt_group(g)
    wt_rank = 0 if wt_present else 1
    return (wt_rank,) + tuple(norm) + (str(g).upper(),)

def _order_x_groups(groups):
    return sorted(groups, key=_hier_sort_key)

def _choose_ref_group(order):
    for g in order:
        if _is_wt_group(g):
            return g
    return order[0] if order else None

def _order_hue_groups(hues):
    hp = globals().get("HUE_PRIORITY", ["Female", "Male", "F", "M", "ALL", "UNASSIGNED"])
    hp_lower = [p.lower() for p in hp]
    def _prio(h):
        u = str(h).strip()
        try:
            return (0, hp_lower.index(u.lower()), u.upper())
        except ValueError:
            return (1, u.upper())
    return sorted([h for h in hues if h is not None], key=_prio)

# compute initial ordered_x from data
ordered_x = _order_x_groups(long_df["XGroup"].dropna().unique().tolist())

# -----------------------
# 3) Controls (left column: groups & colors)
# -----------------------
named_defaults = [
    "dodgerblue", "red", "green", "orange", "purple",
    "brown", "pink", "gray", "olive", "cyan", "slategrey","peachpuff", "mediumpurple", "mediumblue", "limegreen", "deeppink"
]

x_checks, x_colors = {}, {}
group_rows = []
for i, g in enumerate(ordered_x):
    chk = widgets.Checkbox(value=True, description=g, indent=False, layout=widgets.Layout(width="260px"))
    col = widgets.Text(value=named_defaults[i % len(named_defaults)],
                       layout=widgets.Layout(width="120px"))
    x_checks[g] = chk
    x_colors[g] = col
    group_rows.append(
        widgets.HBox(
            [chk, widgets.Label(""), col],
            layout=widgets.Layout(align_items="center", height="28px")
        )
    )

picker = widgets.VBox(group_rows, layout=widgets.Layout(gap="2px"))

btn_all  = widgets.Button(description="Select all", layout=widgets.Layout(width="140px"))
btn_none = widgets.Button(description="Clear", layout=widgets.Layout(width="140px"))
def _set_all(val):
    for c in x_checks.values():
        c.value = val
btn_all.on_click(lambda _: _set_all(True))
btn_none.on_click(lambda _: _set_all(False))

picker_container = widgets.Box(
    [picker],
    layout=widgets.Layout(
        overflow="auto", max_height="420px",
        border="1px solid #ddd", padding="6px", width="360px"
    )
)

left_col = widgets.VBox(
    [
        widgets.HTML("<b>Groups & Colors</b>"),
        widgets.HBox([btn_all, btn_none], layout=widgets.Layout(gap="8px")),
        picker_container
    ],
    layout=widgets.Layout(width="380px")
)

# -----------------------
# 4) Comparison controls (right column)
# -----------------------
mode_radio = widgets.ToggleButtons(
    options=[("Reference group", "ref"), ("Select Pairs", "pairs")],
    value="ref", description="", style={"button_width":"150px"},
    layout=widgets.Layout(width="320px")
)

ref_dropdown = widgets.Dropdown(
    options=ordered_x, value=_choose_ref_group(ordered_x),
    description="Reference:", layout=widgets.Layout(width="320px")
)

def _pair_label(a,b): return f"{a} ⟷ {b}"
def _pair_value(a,b): return (a,b) if a <= b else (b,a)

pairs_select = widgets.SelectMultiple(
    options=[], value=[], description="Pairs",
    layout=widgets.Layout(width="360px", height="320px")
)

def _selected_x():
    return _order_x_groups([g for g, cb in x_checks.items() if cb.value])

def _pair_sort_key(a, b):
    A = _x_levels(a); B = _x_levels(b)
    L = max(len(A), len(B))
    if len(A) < L: A += [""] * (L - len(A))
    if len(B) < L: B += [""] * (L - len(B))
    first_diff = next((i for i, (x, y) in enumerate(zip(A, B)) if x != y), L)
    prefix = tuple(A[:first_diff])
    return (-first_diff, prefix, tuple(A), tuple(B))

def _update_ref_and_pairs(*_):
    sel = _selected_x()
    ref_dropdown.options = sel or ["—"]
    if sel:
        if ref_dropdown.value not in sel:
            ref_dropdown.value = _choose_ref_group(sel)
    else:
        ref_dropdown.value = None

    opts = []
    for a, b in itertools.combinations(sel, 2):
        lbl = _pair_label(a, b)
        val = _pair_value(a, b)
        opts.append((lbl, val))
    opts.sort(key=lambda kv: _pair_sort_key(*kv[1]))
    pairs_select.options = opts

for cb in x_checks.values():
    cb.observe(_update_ref_and_pairs, names="value")
_update_ref_and_pairs()

plot_btn = widgets.Button(description="Plot", button_style="primary",
                          layout=widgets.Layout(width="160px"))
save_btn = widgets.Button(description="Save Plots", button_style="success",
                          layout=widgets.Layout(width="160px"))

right_col = widgets.VBox(
    [
        widgets.HTML("<b>Statistical comparisons</b>"),
        mode_radio,
        ref_dropdown,
        pairs_select,
        widgets.HBox([plot_btn, save_btn], layout=widgets.Layout(gap="8px"))
    ],
    layout=widgets.Layout(width="360px")
)

# -----------------------
# 5) Labels for stats/legend
# -----------------------
def _grouping_label(which="X"):
    if which.lower().startswith("x"):
        cols = globals().get("selected_group_cols_x", [])
        default = "XGroup"
    else:
        cols = globals().get("selected_group_cols_hue", [])
        default = "HueGroup"
    cols = [str(c).strip() for c in (cols or []) if str(c).strip()]
    return " | ".join(cols) if cols else default

# -----------------------
# 6) Stats helpers (ANOVA with Hue)
# -----------------------
def _fmt_p(p):
    if not np.isfinite(p): return "n/a"
    return f"p = {p:.3f}" if p >= 0.001 else "p < 0.001"

def _anova_subset(df):
    out = {"p_x": np.nan, "p_h": np.nan, "p_int": np.nan, "n_h": 0, "ok": False, "err": None}
    d = df.dropna(subset=["value","XGroup"])
    if d.empty or d["XGroup"].nunique() < 2:
        out["err"] = "Too few groups"; return out
    n_h = d["HueGroup"].nunique(dropna=True); out["n_h"] = n_h
    try:
        if n_h >= 2:
            model = ols('value ~ C(XGroup) + C(HueGroup) + C(XGroup):C(HueGroup)', data=d).fit()
            an = sm.stats.anova_lm(model, typ=2)
            out["p_x"]   = float(an.loc['C(XGroup)','PR(>F)'])
            out["p_h"]   = float(an.loc['C(HueGroup)','PR(>F)'])
            out["p_int"] = float(an.loc['C(XGroup):C(HueGroup)','PR(>F)'])
            out["ok"] = True
        else:
            model = ols('value ~ C(XGroup)', data=d).fit()
            out["p_x"] = float(model.f_pvalue); out["ok"] = True
    except Exception as e:
        out["err"] = str(e)
    return out

def _stats_text(dfm, x_label, hue_label, *, mode="ref", ref_group=None, pair_list=None):
    df = dfm.dropna(subset=["value"]).copy()
    g_n = df["XGroup"].nunique(dropna=True)
    h_n = df["HueGroup"].nunique(dropna=True)

    if mode == "pairs" and pair_list:
        lines = ["Selected pairwise ANOVA tests:"]
        for a,b in pair_list:
            sub = df[df["XGroup"].isin([a,b])]
            res = _anova_subset(sub)
            if not res["ok"]:
                lines.append(f"{a} vs {b}: {res['err'] or 'failed'}"); continue
            if res["n_h"] >= 2:
                lines.append(
                    f"{a} vs {b} (Two-way: {x_label}, {hue_label})  "
                    f"{x_label}: {_fmt_p(res['p_x'])} | {hue_label}: {_fmt_p(res['p_h'])} | "
                    f"{x_label}×{hue_label}: {_fmt_p(res['p_int'])}"
                )
            else:
                lines.append(f"{a} vs {b} (One-way {x_label}): {_fmt_p(res['p_x'])}")
        return "\n".join(lines)

    def fmt(p): return _fmt_p(p)
    if g_n == 2 and h_n <= 1:
        g1, g2 = sorted(df["XGroup"].unique())
        v1 = df[df["XGroup"] == g1]["value"].dropna()
        v2 = df[df["XGroup"] == g2]["value"].dropna()
        if len(v1) > 1 and len(v2) > 1:
            p = pg.ttest(v1, v2, paired=False)["p-val"].values[0]
            return f"t-test ({x_label}): {fmt(p)}\n{g1} vs {g2}"
        return "t-test: not enough data"

    if g_n >= 2 and h_n >= 2:
        try:
            model = ols('value ~ C(XGroup) + C(HueGroup) + C(XGroup):C(HueGroup)', data=df).fit()
            an = sm.stats.anova_lm(model, typ=2)
            return (
                "Two-way ANOVA\n"
                f"{x_label}: {fmt(float(an.loc['C(XGroup)','PR(>F)']))}\n"
                f"{hue_label}: {fmt(float(an.loc['C(HueGroup)','PR(>F)']))}\n"
                f"{x_label}×{hue_label}: {fmt(float(an.loc['C(XGroup):C(HueGroup)','PR(>F)']))}"
            )
        except Exception as e:
            return f"ANOVA failed: {e}"

    if g_n >= 2:
        try:
            model = ols('value ~ C(XGroup)', data=df).fit()
            return f"One-way ANOVA ({x_label}): {fmt(float(model.f_pvalue))}"
        except Exception as e:
            return f"One-way ANOVA failed: {e}"
    return "Too few groups for stats"

# -----------------------
# 7) Plotting helpers
# -----------------------
def _p_to_stars(p):
    if not np.isfinite(p): return ""
    if p < 1e-4: return "****"
    if p < 1e-3: return "***"
    if p < 1e-2: return "**"
    if p < 5e-2: return "*"
    return ""

def _dot_palette(hues):
    hues = list(hues)
    if len(hues) == 0: return {}
    if len(hues) == 1: return {hues[0]: "black"}
    if len(hues) == 2: return {hues[0]: "white", hues[1]: "black"}
    defaults = plt.rcParams.get('axes.prop_cycle', None)
    colors = defaults.by_key()['color'] if defaults else ["C0","C1","C2","C3","C4","C5","C6","C7","C8","C9"]
    return {h: colors[i % len(colors)] for i, h in enumerate(hues)}

def _draw_bracket(ax, x1, x2, y, h, text):
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1, c="black", zorder=5)
    ax.text((x1+x2)/2, y+h, text, ha="center", va="bottom", fontsize=16, fontweight="bold")

def _plot_metric_clean(df_metric, variable, x_color_map, *, mode="ref", ref_group=None, pair_list=None, return_fig=False):
    dfm = df_metric.copy()
    order = _order_x_groups(dfm["XGroup"].dropna().unique().tolist())
    if not order:
        return None
    if (not ref_group) or (ref_group not in order):
        ref_group = _choose_ref_group(order)

    x_label_name   = _grouping_label("X")
    hue_label_name = _grouping_label("Hue")

    # Bars
    width = max(2, 1 * len(order))
    height = 4.0
    fig, (ax_plot, ax_text) = plt.subplots(
        1, 2, figsize=(width/0.8, height), gridspec_kw={'width_ratios': [3, 1]}
    )

    bar_palette = [x_color_map.get(g, "tab:blue") for g in order]
    sns.barplot(
        data=dfm, x="XGroup", y="value",
        order=order, ci=None, alpha=ALPHA, ax=ax_plot, palette=bar_palette
    )

    # Hue ordering + dot colors
    raw_hues = dfm["HueGroup"].dropna().unique().tolist()
    hue_levels = _order_hue_groups(raw_hues)
    pal_dots = _dot_palette(hue_levels)

    sns.stripplot(
        data=dfm, x="XGroup", y="value",
        order=order,
        hue="HueGroup",
        hue_order=hue_levels,
        jitter=True, dodge=False, size=7,
        edgecolor="black", linewidth=1,
        palette=pal_dots,
        ax=ax_plot, zorder=3, alpha=ALPHA
    )
    if ax_plot.legend_ is not None:
        ax_plot.legend_.remove()

    if len(hue_levels) >= 2:
        handles = [plt.Line2D([0],[0], marker='o', linestyle='None',
                              markerfacecolor=pal_dots[h], markeredgecolor='black', label=str(h))
                  for h in hue_levels]
        ax_text.legend(handles=handles, title=hue_label_name, loc="upper left", bbox_to_anchor=(0, 0.6))

    y_min, y_max = ax_plot.get_ylim()
    span = (y_max - y_min) if y_max > y_min else 1.0
    bump = 0.06 * span
    data_max = dfm["value"].max() if dfm["value"].notna().any() else y_max

    if mode == "ref" and (ref_group in order):
        ref_vals = dfm[dfm["XGroup"] == ref_group]["value"].dropna().to_numpy()
        for g in order:
            if g == ref_group:
                continue
            vals = dfm[dfm["XGroup"] == g]["value"].dropna().to_numpy()
            if len(vals) >= 2 and len(ref_vals) >= 2:
                try:
                    p = float(pg.ttest(vals, ref_vals, paired=False)["p-val"].values[0])
                except Exception:
                    p = np.nan
                if np.isfinite(p) and p < 0.05:
                    xloc = order.index(g)
                    gmax = dfm[dfm["XGroup"] == g]["value"].max()
                    y_star = (gmax if np.isfinite(gmax) else data_max) + bump
                    ax_plot.text(
                        xloc, y_star, _p_to_stars(p),
                        ha="center", va="bottom", fontsize=16, fontweight="bold"
                    )
                    y_max = max(y_max, y_star + bump)
        ax_plot.set_ylim(y_min, y_max)

    elif mode == "pairs" and pair_list:
        base = (dfm["value"].max() if dfm["value"].notna().any() else y_max) + bump
        step = 0.12 * span
        k = 0
        for a,b in pair_list:
            if (a not in order) or (b not in order):
                continue
            sub = dfm[dfm["XGroup"].isin([a,b])].dropna(subset=["value"])
            if sub["XGroup"].nunique() < 2:
                continue
            res = _anova_subset(sub)
            if res["ok"] and np.isfinite(res["p_x"]) and (res["p_x"] < 0.05):
                x1 = order.index(a); x2 = order.index(b)
                if x1 > x2: x1, x2 = x2, x1
                y_here = base + k * step
                _draw_bracket(ax_plot, x1, x2, y_here, 0.04 * span, _p_to_stars(res["p_x"]))
                y_max = max(y_max, y_here + 0.08 * span)
                k += 1
        ax_plot.set_ylim(y_min, y_max)

    ax_plot.set_title("")
    ax_plot.set_xlabel("")
    ax_plot.set_ylabel(variable)
    plt.setp(ax_plot.get_xticklabels(), rotation=45, ha='right')
    sns.despine(ax=ax_plot)

    ax_text.axis("off")
    ax_text.text(
        0, 1,
        _stats_text(dfm, x_label_name, hue_label_name, mode=mode, ref_group=ref_group, pair_list=pair_list),
        va="top", ha="left", fontsize=12, transform=ax_text.transAxes
    )

    plt.tight_layout()
    return fig if return_fig else plt.show()

# -----------------------
# 8) Actions
# -----------------------
out = widgets.Output()

def _selected_x_and_colors():
    sel = _selected_x()
    color_map = {}
    for g in sel:
        val = x_colors[g].value.strip()
        color_map[g] = val if val else "tab:blue"
    return sel, color_map

def _current_pairs():
    return list(pairs_select.value)

def _run_plots(_=None):
    with out:
        clear_output()
        sel_x, color_map = _selected_x_and_colors()
        if len(sel_x) < 1:
            print("Select at least one X group.");
            return

        mode = mode_radio.value
        if mode == "ref":
            ref = ref_dropdown.value if (ref_dropdown.value in sel_x) else _choose_ref_group(sel_x)
            print(f"Showing X groups: {sel_x}  |  reference for stars: {ref}")
        else:
            pair_list = _current_pairs()
            if not pair_list:
                print(f"Showing X groups: {sel_x}  |  no pairs selected (select at least one).")
                return
            print(f"Showing X groups: {sel_x}  |  pairs: {pair_list}")

        metrics = sorted(long_df["variable"].dropna().unique())
        for metric in metrics:
            subset = long_df[(long_df["variable"] == metric) & (long_df["XGroup"].isin(sel_x))]
            if subset["value"].dropna().empty:
                print(f"Skipping {metric} — no data for selected X groups.")
                continue
            if mode == "ref":
                _plot_metric_clean(
                    subset, metric,
                    x_color_map={g: color_map[g] for g in sel_x if g in subset['XGroup'].unique()},
                    mode="ref", ref_group=ref
                )
            else:
                _plot_metric_clean(
                    subset, metric,
                    x_color_map={g: color_map[g] for g in sel_x if g in subset['XGroup'].unique()},
                    mode="pairs", pair_list=_current_pairs()
                )

def _save_plots(_=None):
    with out:
        clear_output()
        sel_x, color_map = _selected_x_and_colors()
        if len(sel_x) < 1:
            print("Select at least one X group.");
            return

        mode = mode_radio.value
        ref = ref_dropdown.value if (mode == "ref") else None
        pair_list = _current_pairs() if (mode == "pairs") else None
        if mode == "pairs" and not pair_list:
            print("Select at least one pair before saving.");
            return

        os.makedirs("BEAM_metric_comparisons", exist_ok=True)
        saved = 0

        metrics = sorted(long_df["variable"].dropna().unique())
        for metric in metrics:
            subset = long_df[(long_df["variable"] == metric) & (long_df["XGroup"].isin(sel_x))]
            if subset["value"].dropna().empty:
                continue
            fig = _plot_metric_clean(
                subset, metric,
                x_color_map={g: color_map[g] for g in sel_x if g in subset['XGroup'].unique()},
                mode=mode, ref_group=ref, pair_list=pair_list, return_fig=True
            )
            safe = metric.replace(" ", "_").replace("/", "-")
            fig.savefig(f"BEAM_metric_comparisons/{safe}.pdf", dpi=300, bbox_inches="tight")
            plt.close(fig); saved += 1

        if saved == 0:
            print("No figures to save.");
            return
        zipname = f"BEAM_metric_comparisons_{int(time.time())}.zip"
        shutil.make_archive(zipname.replace(".zip",""), 'zip', "BEAM_metric_comparisons")
        if colab_files is not None:
            colab_files.download(zipname)
        print(f"Saved {zipname}")

plot_btn.on_click(_run_plots)
save_btn.on_click(_save_plots)

# -----------------------
# 9) Assemble compact UI (two columns)
# -----------------------
def _toggle_controls(*_):
    if mode_radio.value == "ref":
        ref_dropdown.layout.display = ""
        pairs_select.layout.display = "none"
    else:
        ref_dropdown.layout.display = "none"
        pairs_select.layout.display = ""
_toggle_controls()
mode_radio.observe(lambda _: _toggle_controls(), names="value")

row = widgets.HBox(
    [left_col, right_col],
    layout=widgets.Layout(
        justify_content="flex-start",
        align_items="flex-start",
        gap="16px",
        width="auto"
    )
)

ui = widgets.VBox(
    [
        widgets.HTML("<h3 style='margin-bottom:6px'>BEAM metrics plots</h3>"),
        row,
        out
    ],
    layout=widgets.Layout(width="auto")
)

display(ui)

# Auto-run once
_run_plots()

In [ ]:
#@title PLOTS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

OMEGA = globals().get("OMEGA", 2 * np.pi / 24.0)
LIGHTS_OFF_HOUR = globals().get("LIGHTS_OFF_HOUR", 18)

def zt_unsigned_to_signed(zt_hours):
    """ZT [0..24) -> signed ZT [-12, +12)."""
    zt = np.asarray(zt_hours, dtype=float)
    return ((zt + 12) % 24) - 12

def cosinor_func(t, mesor, amplitude, acrophase):
    t = np.asarray(t, dtype=float)
    return mesor + amplitude * np.cos(OMEGA * t + acrophase)

# line styles per genotype
LS_MAP = {
    "WT":   "-",
    "Het":  "--",
    "Hom":  ":",
    "Hemi": (0, (6, 2, 1, 2)),  # dash-dot
}

def _color_for_gene_genotype(gene, geno):
    """
    Pull colors from the grouping widget (x_colors) if possible.
    We try to match:
      'GENE | GENO'  (if XGroup was [Gene, Genotype$] or similar)
      'GENO'
      anything that contains both GENE and GENO as substrings.
    Falls back to a genotype-based palette if no match.
    """
    gene_up = str(gene).strip().upper()
    geno_up = str(geno).strip().upper()

    if "x_colors" in globals():
        # exact GENE | GENO
        target = f"{gene_up} | {geno_up}"
        for key, w in x_colors.items():
            label = str(key).strip().upper()
            if label == target:
                val = getattr(w, "value", None)
                if val:
                    return val.strip()

        # exact GENO
        for key, w in x_colors.items():
            label = str(key).strip().upper()
            if label == geno_up:
                val = getattr(w, "value", None)
                if val:
                    return val.strip()

        # contains GENE and GENO
        for key, w in x_colors.items():
            label = str(key).strip().upper()
            if (gene_up in label) and (geno_up in label):
                val = getattr(w, "value", None)
                if val:
                    return val.strip()

    # fallback: genotype-based palette
    palette = sns.color_palette("tab10")
    idx_map = {"WT": 0, "HET": 1, "HOM": 2, "HEMI": 3}
    idx = idx_map.get(geno_up, 4)
    return palette[idx % len(palette)]

def plot_gene_cosinor_grouped(
    BEAM_data,
    lights_off_hour=LIGHTS_OFF_HOUR,
    metric_col="z_WTref",
    show_points=True
):
    """
    For each Gene:
      - Compute per-mouse hourly means of metric_col in ZT space.
      - Compute group mean per (Gene × Genotype_norm × ZT_hr).
      - Fit a 24h cosinor to the group mean.
      - Plot signed ZT [-12..12] with:
          * cosinor line (WT solid, Het dashed, Hom dotted, Hemi dash-dot)
          * mean points at each hour (group average)
    Colors:
      * pulled from x_colors based on Gene+Genotype if possible.
    """
    from scipy.optimize import curve_fit

    needed = ["datetime", "Gene", "Genotype_norm", "Mouse_ID", metric_col]
    missing = [c for c in needed if c not in BEAM_data.columns]
    if missing:
        raise RuntimeError(f"BEAM_data is missing: {missing}")

    df = BEAM_data[needed].copy()
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    df = df[df["datetime"].notna()].copy()
    df = df[np.isfinite(df[metric_col])].copy()

    # ZT hour
    df["ZT_hr"] = ((df["datetime"].dt.hour - lights_off_hour) % 24).astype(int)

    # per-mouse hourly mean
    per_mouse_hour = (
        df
        .groupby(["Gene", "Genotype_norm", "Mouse_ID", "ZT_hr"], observed=True)[metric_col]
        .mean()
        .reset_index()
    )

    # group mean (across mice)
    grp = (
        per_mouse_hour
        .groupby(["Gene", "Genotype_norm", "ZT_hr"], observed=True)[metric_col]
        .mean()
        .reset_index()
        .rename(columns={metric_col: "group_mean"})
    )

    genes = grp["Gene"].dropna().unique().tolist()
    if not genes:
        raise RuntimeError("No Gene values found in BEAM_data / grouped table.")

    # cosinor fits per Gene × Genotype_norm
    fit_rows = []
    for (gene, geno), sub in grp.groupby(["Gene", "Genotype_norm"], observed=True):
        t = sub["ZT_hr"].to_numpy(dtype=float)
        y = sub["group_mean"].to_numpy(dtype=float)

        if len(t) < 4 or np.nanstd(y) == 0:
            fit_rows.append({
                "Gene": gene,
                "Genotype_norm": geno,
                "MESOR": np.nan,
                "Amplitude": np.nan,
                "Acrophase_ZT_unsigned": np.nan,
                "Acrophase_ZT_signed": np.nan,
                "Cosinor_R2": np.nan,
                "N_hours_used": len(t),
            })
            continue

        guess = [np.nanmean(y), (np.nanmax(y) - np.nanmin(y)) / 2.0, 0.0]
        try:
            params, _ = curve_fit(cosinor_func, t, y, p0=guess, maxfev=20000)
            mesor, amplitude, phi = map(float, params)
            if amplitude < 0:
                amplitude = -amplitude
                phi += np.pi
            y_fit = cosinor_func(t, mesor, amplitude, phi)
            ss_res = float(np.nansum((y - y_fit) ** 2))
            ss_tot = float(np.nansum((y - np.nanmean(y)) ** 2))
            r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else np.nan

            peak_zt_u = ((-phi) % (2 * np.pi)) / OMEGA
            peak_zt_s = zt_unsigned_to_signed(peak_zt_u)
        except Exception:
            mesor = amplitude = r2 = peak_zt_u = peak_zt_s = np.nan

        fit_rows.append({
            "Gene": gene,
            "Genotype_norm": geno,
            "MESOR": mesor,
            "Amplitude": amplitude,
            "Acrophase_ZT_unsigned": peak_zt_u,
            "Acrophase_ZT_signed": peak_zt_s,
            "Cosinor_R2": r2,
            "N_hours_used": len(t),
        })

    fits = pd.DataFrame(fit_rows).sort_values(["Gene", "Genotype_norm"]).reset_index(drop=True)

    # plotting
    sns.set_style("white")
    GENO_ORDER = ["WT", "Het", "Hom", "Hemi"]

    for gene in sorted(genes):
        g_grp = grp[grp["Gene"] == gene]
        if g_grp.empty:
            continue

        gene_fit = fits[fits["Gene"] == gene]

        fig, ax = plt.subplots(figsize=(8, 4))
        # shade ZT 0–12 (night on unsigned axis)
        ax.axvspan(0, 12, color="0.9", zorder=-1)

        # draw each genotype in fixed order
        for geno in [g for g in GENO_ORDER if g in g_grp["Genotype_norm"].unique()]:

            sub = g_grp[g_grp["Genotype_norm"] == geno].sort_values("ZT_hr")
            if sub.empty:
                continue

            # group mean per hour -> points
            x_u = sub["ZT_hr"].to_numpy(dtype=float)             # 0..23
            x_s = zt_unsigned_to_signed(x_u)                     # -12..12
            y_mean = sub["group_mean"].to_numpy(dtype=float)

            order = np.argsort(x_s)
            x_s = x_s[order]
            y_mean = y_mean[order]

            color = _color_for_gene_genotype(gene, geno)
            ls = LS_MAP.get(geno, "-")

            # points = group means per hour
            if show_points:
                ax.plot(
                    x_s,
                    y_mean,
                    "o",
                    color=color,
                    markersize=6,
                    alpha=0.8,
                    label=None
                )

            # cosinor curve using fit params
            row_fit = gene_fit[gene_fit["Genotype_norm"] == geno]
            if not row_fit.empty and np.isfinite(row_fit["MESOR"].iloc[0]):
                mesor = float(row_fit["MESOR"].iloc[0])
                amp = float(row_fit["Amplitude"].iloc[0])
                peak_u = float(row_fit["Acrophase_ZT_unsigned"].iloc[0])
                peak_s = float(row_fit["Acrophase_ZT_signed"].iloc[0])
                if np.isfinite(peak_u):
                    phi = (-peak_u * OMEGA) % (2 * np.pi)
                else:
                    phi = 0.0

                t_fit_u = np.linspace(0, 23.999, 400)
                y_fit = cosinor_func(t_fit_u, mesor, amp, phi)
                x_fit_s = zt_unsigned_to_signed(t_fit_u)

                m_neg = x_fit_s < 0
                m_pos = ~m_neg
                label_done = False
                if m_neg.any():
                    idx = np.argsort(x_fit_s[m_neg])
                    ax.plot(
                        x_fit_s[m_neg][idx],
                        y_fit[m_neg][idx],
                        color=color,
                        linestyle=ls,
                        linewidth=2.0,
                        label=f"{geno}"
                    )
                    label_done = True
                if m_pos.any():
                    idx = np.argsort(x_fit_s[m_pos])
                    ax.plot(
                        x_fit_s[m_pos][idx],
                        y_fit[m_pos][idx],
                        color=color,
                        linestyle=ls,
                        linewidth=2.0,
                        label=None if label_done else f"{geno}"
                    )

                if np.isfinite(peak_s):
                    ax.axvline(
                        peak_s,
                        color=color,
                        linestyle=":",
                        linewidth=1.5,
                        alpha=0.9
                    )
            else:
                # fallback: connect means
                ax.plot(
                    x_s,
                    y_mean,
                    color=color,
                    linestyle=ls,
                    linewidth=2.0,
                    label=f"{geno}"
                )

        ax.set_xlim(-12, 12)
        ax.set_xticks([-12, -6, 0, 6, 12])
        ax.set_xlabel("ZT")
        ax.set_ylabel(f"Activity (Z)")
        ax.set_title(f"Gene: {gene}")
        sns.despine(ax=ax)

        # clean legend
        handles, labels = ax.get_legend_handles_labels()
        seen = set()
        h2, l2 = [], []
        for h, l in zip(handles, labels):
            if l not in seen:
                seen.add(l); h2.append(h); l2.append(l)
        if h2:
            ax.legend(h2, l2, title="Genotype", frameon=False, loc="upper right")

        plt.tight_layout()
        plt.show()

    return fits

# ---- run it ----
fits_gene = plot_gene_cosinor_grouped(BEAM_data, metric_col="z_WTref", show_points=True)
